In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

from csv import reader, writer
from io import StringIO

In [ ]:
#This goes through the csv and searches for 'nulls', and '[BLOCK TYPE'
#it creates a list based on these finds and removes the first value
#this is meant to find the first and last values of each block type

def getInds (fileName, flags):

    indexes = []
    i = 0

    with open(fileName, 'r') as read_obj: # uses csv_reader to go through each line

        csv_reader = reader(read_obj)

        for count, row in enumerate(csv_reader):

            try: #handles null values

                blockType = row[0]

            except IndexError:

                blockType = 'null'

            if blockType in flags: #== '[BLOCK TYPE' or blockType == 'null' or blockType == '': #checks for these values

                indexes.append(count)

        indexes.pop(0) # removes first null value

        return (indexes)

In [ ]:
def calcDiffAndMake2D (indexes):

        hopper = 0
        arrWithDiffs = []

        for count, val in enumerate(indexes):

            cols = []

            if((count)%2 != 0): #if the iteration is odd

                cols.append(indexes[count-1]) # the first value is the same
                cols.append(val - indexes[count-1] -1) #the second value is the 'val' (second index) minus the (first index (minus 1)) to get the difference

                arrWithDiffs.append(cols) #This appends the columns into a new array which is now a 2D array

        return (arrWithDiffs)


In [ ]:
def makeDfDict (fileName): # this makes a dictionary of dataframes based off of a csv file name

    dfs = {}
    bumpers = ['[BLOCK TYPE', 'null', ''] #these are the values that determine the start and end of the block types - '' was an important add

    arrWithDiffs = calcDiffAndMake2D(getInds(fileName, bumpers)) # finds the indexes with the file and the bumpers, then calculates the difference and returns a 2D array
    print(arrWithDiffs) #good troubleshooter

    for i in range(len(arrWithDiffs)):

        dfs[i] = pd.read_csv(fileName, skiprows=arrWithDiffs[i][0], nrows=arrWithDiffs[i][1]) #splits up the dfs according to the index and difference of the arr

    return (dfs)

In [ ]:
dfDict = makeDfDict('***.csv') #2 total

[[3, 2], [7, 5]]
[[3, 5], [10, 6], [18, 5]]
[[3, 108], [113, 9], [124, 8], [134, 14], [150, 1046], [1198, 24]]
[[3, 333]]
[[3, 8], [13, 18], [33, 398], [433, 1749], [2184, 338], [2524, 3], [2529, 6]]
[[3, 2], [7, 2], [11, 479], [492, 4]]


In [ ]:
# function that changes specific columns of block types
def changeAlarms (dfDict, columnName, replacee, replacer, enableChecker, ignore):

    for i in dfDict: #iterates through the df dictionaries

        if enableChecker: #if a column only needs to be changed based on a conditional, set to true

            for count, val in enumerate(dfDict[i][columnName]): #loops through the df

                if (dfDict[i]['ALARM ENABLE'][count] == 'ENABLE'):
                    dfDict[i][columnName][count] =  replacer

        elif ignore != '': #if there is a value for ignore...

            for count in range(len(dfDict[i])):

                if (dfDict[i]['[BLOCK TYPE'][count] != ignore and dfDict[i]['[BLOCK TYPE'][count] != '!A_NAME'): # if the value is not the ignore and is not the abbreviated title
                        dfDict[i][columnName][count] =  replacer # then it goes ahead and replaces


        else: #if the two conditionals above aren't the case, else...

            dfDict[i][columnName].replace({replacee: replacer}, inplace=True) #replace the 'replacee' with the 'replacer


In [ ]:
#changes the alarms with the associated variables

changeAlarms(dfDictBatt, "SECURITY AREA 1", "NONE", "UTL_MAITENANCE", False, '')

changeAlarms(dfDictComms, "SECURITY AREA 1", "NONE", "UTL_MAITENANCE", False, '')

changeAlarms(dfDictAlarms, "SECURITY AREA 1", "NONE", "UTL_MAITENANCE", True, '')

changeAlarms(dfDictOvr, "SECURITY AREA 1", "NONE", "UTL_LS_OVR", False, '')

changeAlarms(dfDictValve, "SECURITY AREA 1", "NONE", "UTL_VALVE", False, 'ETR')

changeAlarms(dfDictAck, "SECURITY AREA 1", "NONE", "UTL_ALARM", False, '')

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()


In [ ]:
#dfDictValve[5]['SECURITY AREA 1']

0        A_SA1
1    UTL_VALVE
2    UTL_VALVE
Name: SECURITY AREA 1, dtype: object

In [ ]:
#this function checks for duplicate values and deletes the second+ instance from the df
def delDubs (dfDict, tags):

    columnName = 'TAG'
    abbrName = 'A_TAG'

    for i in range(len(dfDict)): #iterates through the dfs

      for val in dfDict[i][columnName]: #iterates through each tag value

          if val in tags: #if the value is already tagged and is not the abbreviated name...

              dfDict[i].drop(dfDict[i].index[(dfDict[i][columnName] == val)],axis=0,inplace=True) #drop the tag in place (so no blank row)

          elif val != abbrName:

              tags.append(val) #otherwise, add the novel value to tags

    return tags


In [ ]:
tags = [] #starts with an empty list of tags

tagsWithoutDuplicates = delDubs(dfDictAck, delDubs(dfDictValve, delDubs(dfDictOvr, delDubs(dfDictAlarms, delDubs(dfDictComms, delDubs(dfDictBatt, tags))))))
#runs the function with all the dfDicts and passes the tags through each


In [ ]:
print (tagsWithoutDuplicates)

#tagsWithoutDuplicates[:] = [x for x in tagsWithoutDuplicates if x != 'A_TAG']

contains_duplicates = any(tagsWithoutDuplicates.count(element) > 1 for element in tagsWithoutDuplicates) #checks if there's any duplicates in the
print(contains_duplicates)

['PCIP_PLC_BATTERY_LOW', 'CIP_PLC_BATTERY', 'AC7UTL_PLC_BATTERY', 'CCCIP_PLC_BATTERY', 'PCIP_PLC_BATTERY_LOW_ALM', 'AC7UTL1_WATER_PLC_COMMS_AI', 'AC7UTL1_UTL_PLC_COMMS_AI', 'AC7UTL1_CIP_PLC_COMMS_AI', 'AC7UTL1_CC_CIP_PLC_COMMS_AI', 'AC7UTL1_UTL_PLC_COMMS', 'AC7UTL1_WATER_PLC_COMMS', 'AC7UTL1_CIP_PLC_COMMS', 'AC7UTL1_ODBC_COMMS', 'AC7UTL1_CC_CIP_PLC_COMMS', 'AC7UTL1_UTL_PLC_COMMS_EV', 'AC7UTL1_CIP_PLC_COMMS_EV', 'AC7UTL1_CC_CIP_PLC_COMMS_EV', 'AC7UTL1_WATER_PLC_COMMS_EV', 'WTR_PLC_DLR_RING_SPEED', 'RO1_DLR_RING_SPEED', 'PWT2_DLR_RING_SPEED', 'PWT1_DLR_RING_SPEED', 'WTR_PLC_DLR_RING_SUP', 'WTR_DLR_STATUS', 'WTR_DLR_TOPOLOGY', 'ETAP_DLR_RING_SPEED', 'WTR_DLR_RING_FLTS', '70A_AIT_015', '54205_AIT_165', '54205_AIT_164', '54205_AIT_163', '54205_AIT_162', '54205_AIT_155', '54205_AIT_154', '54205_AIT_153', '54205_AIT_152', '54205_AIT_125', '54205_AIT_124', '54205_AIT_123', '54205_AIT_122', '54205_AIT_115', '54205_AIT_114', '54205_AIT_113', '54205_AIT_112', '18002_PIT_008_LAV', '18002_PIT_008_L

In [ ]:
def iterthru (d):
    for i in d:
        print(d[i])

iterthru(dfDictValve)

  [BLOCK TYPE                 TAG  ...   SHELVE ENABLE      SHELVE POLICY]
0     !A_NAME               A_TAG  ...  A_IALMSHLVENAB  A_ALMSHELVEPOLICY!
1          AI     260_XV_115_HIGH  ...         DISABLE                 NaN
2          AI   260_XV_10306_HIGH  ...         DISABLE                 NaN
3          AI      260_XV_115_LOW  ...         DISABLE                 NaN
4          AI    260_XV_10306_LOW  ...         DISABLE                 NaN
5          AI          260_XV_115  ...         DISABLE                 NaN
6          AI  553_WQI_025_MONDAY  ...         DISABLE                 NaN
7          AI  552_WQI_035_MONDAY  ...         DISABLE                 NaN

[8 rows x 74 columns]
   [BLOCK TYPE  ...      SHELVE POLICY]
0      !A_NAME  ...  A_ALMSHELVEPOLICY!
1           DI  ...                 NaN
2           DI  ...                 NaN
5           DI  ...                 NaN
6           DI  ...                 NaN
7           DI  ...                 NaN
8           DI  ...   

In [ ]:
#this function inputs the dfs back into a csv that looks like it did when it was uploaded
def makePretty(dfHeader, dfDict, outName):

      #these four lines look at the first rows and insert them in an out csv
      dfHeader1 = pd.read_csv(dfHeader, nrows=0)
      dfHeader2 = pd.read_csv(dfHeader, skiprows = 1, nrows=0)
      dfHeader1.drop(dfHeader1.filter(regex="Unname"),axis=1, inplace=True) #takes care of 'Unanamed' gettting inserted in empy cells
      dfHeader2.drop(dfHeader2.filter(regex="Unname"),axis=1, inplace=True)

      dfHeader1.to_csv(outName, index = False)
      dfHeader2.to_csv(outName, index = False, mode = 'a') #mode a appends rather than writes over


      with open(outName, 'a') as f_object: #this adds a blank row for spacing
          writer_object = writer(f_object)
          writer_object.writerow('')
          f_object.close()


      for i in dfDict: #iterates through the dfs and writes as a csv with a space inbetween
          dfDict[i].drop(dfDict[i].filter(regex="Unname"),axis=1, inplace=True)
          dfDict[i].to_csv(outName, index = False, mode = 'a')

          with open(outName, 'a') as f_object:
              writer_object = writer(f_object)
              writer_object.writerow('')
              f_object.close()

      with open(outName, 'a') as f_object: #last line of the code
          writer_object = writer(f_object)
          writer_object.writerow(['[-------------------------------------------------End of Block List-------------------------------------------------]'])
          f_object.close()

In [ ]:
makePretty('***.csv', dfDictBatt, '***.csv')


In [ ]:
#This function looks for duplicates in the code and deletes them on the second+ instance
"""
def checkAndDelDubs (d1, d2, d3, d4, d5, d6):
    tags = []
    dubs = []

    for i in range(len(d1)): # batt
        for val in d1[i]['TAG']:
            tags.append(val)

    for i in range(len(d2)): #comm
        for val in d2[i]['TAG']:
            if val in tags:
                s = "d2: ", i, ", val: ", val
                dubs.append(s)
                if val != 'A_TAG': d2[i].drop(d2[i].index[(d3[i]["TAG"] == val)],axis=0,inplace=True)
            else:
                tags.append(val)

    for i in range(len(d3)): #alarm
        for val in d3[i]['TAG']:
            if val in tags:
                s = "d3: ", i, ", val: ", val
                dubs.append(s)
                if val != 'A_TAG': d3[i].drop(d3[i].index[(d3[i]["TAG"] == val)],axis=0,inplace=True)
            else:
                tags.append(val)

    for i in range(len(d4)): #ovr
        for val in d4[i]['TAG']:
            if val in tags:
                s = "d4: ", i, ", val: ", val
                dubs.append(s)
                if val != 'A_TAG': d4[i].drop(d4[i].index[(d4[i]["TAG"] == val)],axis=0,inplace=True)
            else:
                tags.append(val)

    for i in range(len(d5)): #vlv
        for val in d5[i]['TAG']:
            if val in tags:
                s = "d5: ", i, ", val: ", val
                dubs.append(s)
                if val != 'A_TAG': d5[i].drop(d5[i].index[(d5[i]["TAG"] == val)],axis=0,inplace=True)
            else:
                tags.append(val)


    for i in range(len(d6)): #ack
        for val in d6[i]['TAG']:
            if val in tags:
                s = "d6: ", i, ", val: ", val
                dubs.append(s)
                if val != 'A_TAG': d6[i].drop(d6[i].index[(d6[i]['TAG'] == val)],axis=0,inplace=True)
            else:
                tags.append(val)
    print("dubs: ", dubs)
    print(tags)
  """

In [ ]:
#less pretty but just as functional way to change the alarms

"""
for i in dfDictBatt:
    dfDictBatt[i]["SECURITY AREA 1"].replace({"NONE": "UTL_MAITENANCE"}, inplace=True)


for i in dfDictComms:
    dfDictComms[i]["SECURITY AREA 1"].replace({"NONE": "UTL_MAITENANCE"}, inplace=True)


for i in range(len(dfDictAlarms)): #replaces the sa1 to utl_maitenance if the alarm is enabled
    for j in range(len(dfDictAlarms[i])):
        if (dfDictAlarms[i]['ALARM ENABLE'][j] == 'ENABLE'):
            dfDictAlarms[i]['SECURITY AREA 1'][j] =  "UTL_MAITENANCE"

dfDictOvr[0]["SECURITY AREA 1"].replace({"NONE": "UTL_LS_OVR"}, inplace=True)


for i in range(6):
    dfDictValve[i]["SECURITY AREA 1"].replace({"NONE": "UTL_VALVE"}, inplace=True)

for i in dfDictAck:
    dfDictAck[i]["SECURITY AREA 1"].replace({"NONE": "UTL_ALARM"}, inplace=True)
"""

'\nfor i in dfDictBatt:\n    dfDictBatt[i]["SECURITY AREA 1"].replace({"NONE": "UTL_MAITENANCE"}, inplace=True)\n\n\nfor i in dfDictComms:\n    dfDictComms[i]["SECURITY AREA 1"].replace({"NONE": "UTL_MAITENANCE"}, inplace=True)\n\n\nfor i in range(len(dfDictAlarms)): #replaces the sa1 to utl_maitenance if the alarm is enabled \n    for j in range(len(dfDictAlarms[i])):\n        if (dfDictAlarms[i][\'ALARM ENABLE\'][j] == \'ENABLE\'):\n            dfDictAlarms[i][\'SECURITY AREA 1\'][j] =  "UTL_MAITENANCE"\n\ndfDictOvr[0]["SECURITY AREA 1"].replace({"NONE": "UTL_LS_OVR"}, inplace=True)\n\n\nfor i in range(6):\n    dfDictValve[i]["SECURITY AREA 1"].replace({"NONE": "UTL_VALVE"}, inplace=True)\n\nfor i in dfDictAck:\n    dfDictAck[i]["SECURITY AREA 1"].replace({"NONE": "UTL_ALARM"}, inplace=True)\n'

In [ ]:
#checkAndDelDubs(dfDictBatt, dfDictComms, dfDictAlarms, dfDictOvr, dfDictValve, dfDictAck)

dubs:  [('d2: ', 0, ', val: ', 'A_TAG'), ('d2: ', 1, ', val: ', 'A_TAG'), ('d2: ', 2, ', val: ', 'A_TAG'), ('d3: ', 0, ', val: ', 'A_TAG'), ('d3: ', 1, ', val: ', 'A_TAG'), ('d3: ', 2, ', val: ', 'A_TAG'), ('d3: ', 3, ', val: ', 'A_TAG'), ('d3: ', 4, ', val: ', 'A_TAG'), ('d3: ', 5, ', val: ', 'A_TAG'), ('d4: ', 0, ', val: ', 'A_TAG'), ('d5: ', 0, ', val: ', 'A_TAG'), ('d5: ', 1, ', val: ', 'A_TAG'), ('d5: ', 2, ', val: ', 'A_TAG'), ('d5: ', 3, ', val: ', 'A_TAG'), ('d5: ', 4, ', val: ', 'A_TAG'), ('d5: ', 5, ', val: ', 'A_TAG'), ('d5: ', 6, ', val: ', 'A_TAG'), ('d6: ', 1, ', val: ', 'A_IOAD'), ('d6: ', 2, ', val: ', 'A_IOAD'), ('d6: ', 3, ', val: ', 'A_IOAD')]
['A_TAG', 'PCIP_PLC_BATTERY_LOW', 'A_TAG', 'CIP_PLC_BATTERY', 'AC7UTL_PLC_BATTERY', 'CCCIP_PLC_BATTERY', 'PCIP_PLC_BATTERY_LOW_ALM', 'AC7UTL1_WATER_PLC_COMMS_AI', 'AC7UTL1_UTL_PLC_COMMS_AI', 'AC7UTL1_CIP_PLC_COMMS_AI', 'AC7UTL1_CC_CIP_PLC_COMMS_AI', 'AC7UTL1_UTL_PLC_COMMS', 'AC7UTL1_WATER_PLC_COMMS', 'AC7UTL1_CIP_PLC_COMMS', 'A